In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"
gemini_model = "gemini-2.5-flash-lite"

gpt_system = "You are GPT, a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

claude_system = "You are Claude, a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

google_system = "You are Google, a helpful assistant who tries to find the truth. You have access to the entire internet and can search for information to answer questions. You are very good at finding accurate information and summarizing it."

gpt_message = ["Hi there"]
claude_message = ["Hi"]
google_message = ["Konnichiwa"]

conversation = [{"agent": "gpt", "message": gpt_message[0]},
                {"agent": "claude", "message": claude_message[0]},
                {"agent": "google", "message": google_message[0]}]


def call_conversation(actor, model, system, conversation):
    # message_template = """ 
    # React to this conversation 
    # The conversation so far is as follows:
    # {conversation}
    # Your task is to provide a response to the last message in the conversation as a {actor}.
    # """

    message_template = """ 
    You are {actor}, in conversation with two other agents.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as {actor}.
    """

    messages = [{"role": "system", "content": system}]
    conversation_text = "\n".join([f"{c['agent']}: {c['message']}" for c in conversation])
    messages.append({"role": "user", "content": message_template.format(conversation=conversation_text, actor=actor)})
    
    if actor == "GPT":
        response = openai.chat.completions.create(model=gpt_model, messages=messages)
        conversation.append({"agent": "GPT", "message": response.choices[0].message.content})
    elif actor == "Claude":
        response = anthropic.chat.completions.create(model=claude_model, messages=messages)
        conversation.append({"agent": "Claude", "message": response.choices[0].message.content})
    elif actor == "Google":
        response = gemini.chat.completions.create(model=gemini_model, messages=messages)
        conversation.append({"agent": "Google", "message": response.choices[0].message.content})
    else:
        raise ValueError("Unknown actor")
    
    return response.choices[0].message.content


display(Markdown(f"### GPT:\n{gpt_message[0]}\n"))
display(Markdown(f"### Claude:\n{claude_message[0]}\n"))
display(Markdown(f"### Google:\n{google_message[0]}\n"))

for i in range(5):
    gpt_next = call_conversation("GPT", gpt_model, gpt_system, conversation)
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    
    claude_next = call_conversation("Claude", claude_model, claude_system, conversation)
    display(Markdown(f"### Claude:\n{claude_next}\n"))
    
    google_next = call_conversation("Google", gemini_model, google_system, conversation)
    display(Markdown(f"### Google:\n{google_next}\n"))